####################################### Here we go Again! ####################################### Introducing the concept of Gut Health Score (GHS) where we need to longitudinally understand how the gut microbiota's health improves from the time of pertubation. Through machine learning, we generated a score and through this score we can monitor how the health improves with time. By doing so, we intend to to apply ML beyond the usually classification and biomarker identification for an insightful longitudinal analysis.

In [ ]:
!pip install xgboost lightgbm shap scikit-learn openpyxl matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score, precision_score,
    recall_score, confusion_matrix, brier_score_loss, roc_curve
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import shap

# ─── Reproducibility ───────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ─── Plot style ────────────────────────────────────────────────────────────
matplotlib.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False
})
print('All imports OK — SEED =', SEED)

In [ ]:
# ── For Google Colab: upload files interactively ──────────────────────────
from google.colab import files
print('Please upload Dataset-1.xlsx and Dataset-2.xlsx')
uploaded = files.upload()

# ── Load datasets ──────────────────────────────────────────────────────────
df1 = pd.read_excel('Dataset-1.xlsx')
df2 = pd.read_excel('Dataset-2.xlsx')

print(f'Dataset 1: {df1.shape[0]} samples × {df1.shape[1]} columns')
print(f'Dataset 2: {df2.shape[0]} samples × {df2.shape[1]} columns')
print('\nStatus distribution (Dataset 1):')
print(df1['Status'].value_counts().sort_index())

In [ ]:
# ─── Feature columns (all columns except Genus & Status) ──────────────────
feat_cols_all = [c for c in df1.columns if c not in ['Genus', 'Status']]

# ─── Remove genera that are ALL zeros in Dataset 1 ────────────────────────
# Rationale: all-zero genera have zero variance → zero contribution to any

zero_genera = [c for c in feat_cols_all if df1[c].sum() == 0]
feat_cols   = [c for c in feat_cols_all if c not in zero_genera]

print(f'Original genera   : {len(feat_cols_all)}')
print(f'All-zero genera   : {len(zero_genera)}  → REMOVED')
print(f'Retained genera   : {len(feat_cols)}')
if zero_genera:
    print('\nRemoved genera:', zero_genera[:10], '...' if len(zero_genera)>10 else '')

# ─── CLR Transform function ────────────────────────────────────────────────
def clr_transform(X):
    """Centered Log-Ratio transform for compositional microbiome data.
    Adds a small pseudocount (1e-6) to handle exact zeros."""
    X = X + 1e-6           # pseudocount to avoid log(0)
    log_X = np.log(X)
    geometric_mean = log_X.mean(axis=1, keepdims=True)
    return log_X - geometric_mean

# ─── Apply CLR to Dataset 1 ────────────────────────────────────────────────
X_raw = df1[feat_cols].values
X_clr = clr_transform(X_raw)

# ─── Variance Threshold Filter ─────────────────────────────────────────────
# After CLR, genera with near-zero variance across all samples still add noise.
# VarianceThreshold(0.01) removes features with variance < 0.01 post-CLR.
sel   = VarianceThreshold(threshold=0.01)
X     = sel.fit_transform(X_clr)
feat_kp = np.array(feat_cols)[sel.get_support()]   # kept feature names

print(f'\nAfter variance filter (threshold=0.01): {X.shape[1]} features retained')

# ─── Encode labels ─────────────────────────────────────────────────────────
le      = LabelEncoder()
y       = le.fit_transform(df1['Status'].values)   # T1→0, T2→1, T3→2
classes = le.classes_
print(f'\nEncoding: {dict(zip(classes, le.transform(classes)))}')

In [ ]:
# ─── Model definitions ─────────────────────────────────────────────────────
# class_weight='balanced' handles mild class imbalance
models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=500, random_state=SEED,
        class_weight='balanced', min_samples_leaf=2),

    'SVM': SVC(
        probability=True, kernel='rbf', C=10, gamma='scale',
        random_state=SEED, class_weight='balanced'),

    'XGBoost': XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        use_label_encoder=False, eval_metric='mlogloss',
        random_state=SEED, verbosity=0, subsample=0.8),

    'LightGBM': LGBMClassifier(
        n_estimators=300, num_leaves=15, learning_rate=0.05,
        random_state=SEED, class_weight='balanced',
        verbose=-1, min_child_samples=5),

    'Logistic Regression': LogisticRegression(
        max_iter=3000, random_state=SEED,
        class_weight='balanced', C=0.1,
        solver='saga', penalty='l1'),

    'Neural Network': MLPClassifier(
        hidden_layer_sizes=(64, 32, 16), max_iter=2000,
        random_state=SEED, alpha=0.01),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# ─── Helper: macro-specificity ─────────────────────────────────────────────
def macro_specificity(y_true, y_pred, n_classes=3):
    cm = confusion_matrix(y_true, y_pred)
    specs = []
    for i in range(n_classes):
        TP = cm[i, i]
        FP = cm[:, i].sum() - TP
        TN = cm.sum() - cm[i, :].sum() - cm[:, i].sum() + TP
        specs.append(TN / (TN + FP) if (TN + FP) > 0 else 0)
    return np.mean(specs)

# ─── Run 5-fold CV for all models ──────────────────────────────────────────
print('Running 5-fold CV... (this may take ~1–2 min)')
results = {}
y_bin = label_binarize(y, classes=[0, 1, 2])

for name, model in models.items():
    proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba')
    pred  = proba.argmax(axis=1)
    auc   = roc_auc_score(y_bin, proba, multi_class='ovr', average='macro')
    acc   = accuracy_score(y, pred)
    f1    = f1_score(y, pred, average='macro', zero_division=0)
    prec  = precision_score(y, pred, average='macro', zero_division=0)
    rec   = recall_score(y, pred, average='macro', zero_division=0)
    spec  = macro_specificity(y, pred)
    brier = np.mean([brier_score_loss((y == c).astype(int), proba[:, c]) for c in range(3)])
    results[name] = {
        'AUC': auc, 'Accuracy': acc, 'Macro-F1': f1,
        'Macro-Precision': prec, 'Macro-Recall (Sensitivity)': rec,
        'Macro-Specificity': spec, 'Brier Score': brier,
        'proba': proba, 'pred': pred
    }
    print(f'  {name:<22s}  AUC={auc:.3f}  F1={f1:.3f}  Acc={acc:.3f}  Spec={spec:.3f}')

best_name = max(results, key=lambda n: results[n]['AUC'])
print(f'\n★ Best model: {best_name}  (AUC = {results[best_name]["AUC"]:.3f})')

In [ ]:
# AUC Comparison Bar Chart

model_names = list(models.keys())
aucs   = [results[n]['AUC'] for n in model_names]
colors = ['#2196F3', '#E91E63', '#FF9800', '#4CAF50', '#9C27B0', '#F44336']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(model_names, aucs, color=colors, height=0.6, edgecolor='white')
for bar, v in zip(bars, aucs):
    ax.text(v + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{v:.3f}', va='center', fontsize=10, fontweight='bold')
ax.set_xlim(0, 1.05)
ax.axvline(0.5, ls='--', color='grey', lw=1, alpha=0.6, label='Random = 0.50')
ax.set_xlabel('Macro-AUC (5-fold CV)', fontsize=12)
ax.set_title('Model Comparison – Macro-AUC\n(5-Fold Cross-Validation)', fontsize=13, fontweight='bold')
ax.invert_yaxis()
# Gold border on best
best_idx = model_names.index(best_name)
bars[best_idx].set_edgecolor('gold'); bars[best_idx].set_linewidth(3)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('Fig1_AUC_comparison.png', bbox_inches='tight', dpi=200)
plt.show()

In [ ]:
# Checking the performance with Performance Metrics

metric_keys = ['AUC', 'Accuracy', 'Macro-F1', 'Macro-Precision',
               'Macro-Recall (Sensitivity)', 'Macro-Specificity', 'Brier Score']
tbl = pd.DataFrame(
    {n: {k: round(results[n][k], 3) for k in metric_keys} for n in model_names}
).T.reset_index().rename(columns={'index': 'Model'})
tbl.to_csv('Table1_performance_metrics.csv', index=False)
display(tbl.style.highlight_max(subset=metric_keys[:-1], color='#c8f7c5')
              .highlight_min(subset=['Brier Score'], color='#c8f7c5')
              .set_caption('Table 1 – 5-Fold CV Performance Metrics (Macro-averaged)'))

In [ ]:
# ROC Curves (Best Model)

y_bin      = label_binarize(y, classes=[0, 1, 2])
proba_best = results[best_name]['proba']

fig, ax = plt.subplots(figsize=(7, 6))
colors_roc = ['#2196F3', '#FF9800', '#4CAF50']
for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_bin[:, i], proba_best[:, i])
    auc_i = roc_auc_score(y_bin[:, i], proba_best[:, i])
    ax.plot(fpr, tpr, color=colors_roc[i], lw=2.5, label=f'{cls} (AUC = {auc_i:.3f})')
ax.plot([0, 1], [0, 1], '--', color='grey', lw=1)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
#ax.set_title(f'ROC Curves – {best_name}\n(5-Fold CV Probabilities)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('Fig2_ROC_curves.png', bbox_inches='tight', dpi=200)
plt.show()

In [ ]:
# Through SHAP, we asked ourselves, what are these Top 10 Bacterial Genera Driving T1, T2, T3 Classification
# Given that the best model turned out to be SVM
print(f'Fitting {best_name} on full dataset...')
best_model = models[best_name]
best_model.fit(X, y)

print('Computing SHAP values (KernelExplainer, ~1–2 min)...')
# KernelExplainer is model-agnostic and works for SVM/any classifier
# We summarise background data with k-means to speed up computation
background  = shap.kmeans(X, 30)   # 30 cluster centres as background
explainer   = shap.KernelExplainer(best_model.predict_proba, background)
shap_vals   = explainer.shap_values(X, nsamples=200)   # (n_samples, n_features, n_classes)
shap_arr    = np.array(shap_vals)  # shape: (n_samples, n_features, n_classes)

# T1 = class index 0
shap_t1   = shap_arr[:, :, 0]           # (n_samples, n_features)
mean_abs  = np.abs(shap_t1).mean(axis=0)
top10_idx = np.argsort(mean_abs)[::-1][:10]
top10_names = feat_kp[top10_idx]
top10_vals  = mean_abs[top10_idx]

print('\nTop 10 genera driving T1 classification:')
for i, (g, v) in enumerate(zip(top10_names, top10_vals), 1):
    print(f'  {i:2d}. {g:<30s}  Mean|SHAP| = {v:.5f}')

In [ ]:
print(f'Fitting {best_name} on full dataset...')
best_model = models[best_name]
best_model.fit(X, y)

print('Computing SHAP values (KernelExplainer, ~1–2 min)...')
# KernelExplainer is model-agnostic and works for SVM/any classifier
# We summarise background data with k-means to speed up computation
background  = shap.kmeans(X, 30)   # 30 cluster centres as background
explainer   = shap.KernelExplainer(best_model.predict_proba, background)
shap_vals   = explainer.shap_values(X, nsamples=200)   # (n_samples, n_features, n_classes)
shap_arr    = np.array(shap_vals)  # shape: (n_samples, n_features, n_classes)

# T3 = class index 2
shap_t1   = shap_arr[:, :, 2]           # (n_samples, n_features)
mean_abs  = np.abs(shap_t1).mean(axis=1)
top10_idx = np.argsort(mean_abs)[::-1][:10]
top10_names = feat_kp[top10_idx]
top10_vals  = mean_abs[top10_idx]

print('\nTop 10 genera driving T1 classification:')
for i, (g, v) in enumerate(zip(top10_names, top10_vals), 1):
    print(f'  {i:2d}. {g:<30s}  Mean|SHAP| = {v:.5f}')

In [ ]:
# ─── Figure 3a: Top 10 SHAP bar chart ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5.5))
y_pos = np.arange(10)
bar_colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.85, 10))
ax.barh(y_pos, top10_vals[::-1], color=bar_colors, edgecolor='white', height=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(list(top10_names[::-1]), fontsize=10)
ax.set_xlabel('Mean |SHAP value| – T1 Classification', fontsize=11)
ax.set_title(f'Top 10 Bacterial Genera Driving T1 Classification\n({best_name}, SHAP KernelExplainer)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('Fig3_SHAP_top10_T1.png', bbox_inches='tight', dpi=200)
#plt.show()

# ─── Figure 3b: SHAP summary (beeswarm) ────────────────────────────────────
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_t1, X, feature_names=feat_kp, max_display=15, show=False, plot_type='dot')
plt.title(f'SHAP Summary – T1 Class ({best_name})', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('Fig3b_SHAP_summary.png', bbox_inches='tight', dpi=200)
plt.show()

The moment is now

## Gut Health Probability Score

### Score definition
$$\text{Score}_i = P(T1)_i \times 0 + P(T2)_i \times 0.5 + P(T3)_i \times 1$$

- Score → **0** means microbiota resembles T1 (pre-treatment / baseline)
- Score → **1** means microbiota resembles T3 (6-month improved state)
- Score around **0.5** means an intermediate T2 profile

We apply **isotonic regression calibration** to ensure the predicted probabilities are well-calibrated before computing the score.

In [ ]:
print(f'Calibrating {best_name} with isotonic regression (5-fold)...')
cal_model = CalibratedClassifierCV(best_model, method='isotonic', cv=5)
cal_model.fit(X, y)
proba_cal = cal_model.predict_proba(X)

# Score weights: T1=0, T2=0.5, T3=1
# ─── ADJUSTABLE: change weights here if your clinical interpretation differs
weights = np.array([0.0, 0.5, 1.0])   # [T1_weight, T2_weight, T3_weight]
scores  = proba_cal @ weights

# Add scores to Dataset 1
df1_scored = df1[['Genus', 'Status']].copy()
df1_scored['P(T1)']            = proba_cal[:, 0].round(3)
df1_scored['P(T2)']            = proba_cal[:, 1].round(3)
df1_scored['P(T3)']            = proba_cal[:, 2].round(3)
df1_scored['Gut_Health_Score'] = scores.round(3)
df1_scored['Predicted_Status'] = le.inverse_transform(proba_cal.argmax(axis=1))
df1_scored.to_csv('Table2_Dataset1_probability_scores.csv', index=False)

score_by = df1_scored.groupby('Status')['Gut_Health_Score'].agg(['mean', 'std', 'median', 'min', 'max'])
score_by = score_by.reindex(['T1', 'T2', 'T3'])
score_by.columns = ['Mean', 'SD', 'Median', 'Min', 'Max']
print('\nGut Health Score by Status:')
display(score_by.round(3))

# Interpret trend
m_t1 = score_by.loc['T1','Mean']; m_t2 = score_by.loc['T2','Mean']; m_t3 = score_by.loc['T3','Mean']
if m_t3 > m_t2 > m_t1:
    trend = 'IMPROVING ↑ (T1 → T2 → T3 scores increase)'
elif m_t3 < m_t2 < m_t1:
    trend = 'DEGRADING ↓ (T1 → T2 → T3 scores decrease)'
else:
    trend = 'NON-MONOTONIC (mixed trend)'
print(f'\nTrend detected: {trend}')

In [ ]:
# Gut Health Score Trend

order   = ['T1', 'T2', 'T3']
xlabels = ['T1\n(Baseline)', 'T2\n(3 months)', 'T3\n(6 months)']
palette = {'T1': '#8B0000', 'T2': '#CD853F', 'T3': '#006400'}
means   = [df1_scored[df1_scored['Status'] == s]['Gut_Health_Score'].mean() for s in order]
stds    = [df1_scored[df1_scored['Status'] == s]['Gut_Health_Score'].std()  for s in order]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ─── Left: Violin + scatter + mean line ───────────────────
ax = axes[0]
for i, status in enumerate(order):
    sub   = df1_scored[df1_scored['Status'] == status]['Gut_Health_Score'].values
    parts = ax.violinplot(sub, positions=[i], widths=0.5, showmedians=True, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(palette[status]); pc.set_alpha(0.4)
    parts['cmedians'].set_colors('black'); parts['cmedians'].set_linewidth(2)
    ax.scatter(np.random.normal(i, 0.06, len(sub)), sub,
               color=palette[status], edgecolor='white', s=45, zorder=5, alpha=0.9)
ax.plot(range(3), means, 'k-o', lw=2.5, ms=9, zorder=10, label='Group mean')
ax.fill_between(range(3),
                [m - s for m, s in zip(means, stds)],
                [m + s for m, s in zip(means, stds)],
                alpha=0.15, color='black')
ax.set_xticks([0, 1, 2]); ax.set_xticklabels(xlabels, fontsize=11, fontweight='bold')
ax.set_ylabel('Gut Health Score', fontsize=11, fontweight='bold'); ax.set_ylim(-0.05, 1.15)
#ax.set_title('Gut Microbiota Trend Across Treatment', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# ─── Right: Bar chart mean ± SD ────────────────────
ax2   = axes[1]
bars2 = ax2.bar(order, means, yerr=stds, capsize=6,
                color=[palette[s] for s in order], edgecolor='white', width=0.5,
                error_kw={'elinewidth': 2, 'ecolor': 'dimgrey'})
for bar, v in zip(bars2, means):
    ax2.text(bar.get_x() + bar.get_width() / 2, v + 0.04,
             f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')
ax2.set_xticklabels(xlabels, fontsize=11, fontweight='bold')
ax2.set_ylabel('Mean Gut Health Score ± SD', fontsize=11, fontweight='bold')
#ax2.set_title('Mean Score by Stage\n(Higher = More T3-like microbiota)', fontsize=12, fontweight='bold')
ax2.set_ylim(0, 1.3)

#plt.suptitle(f'Probability Score Trend – {best_name} (Calibrated)\n'
             #f'Score = P(T3)×1.0 + P(T2)×0.5 + P(T1)×0.0',
             #fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('Fig4_trend_scores.png', bbox_inches='tight', dpi=200)


plt.show()

In [ ]:
order   = ['T1', 'T2', 'T3']
xlabels = ['T1\n(Baseline)', 'T2\n(3 months)', 'T3\n(6 months)']
palette = {'T1': '#4E79A7', 'T2': '#F28E2B', 'T3': '#59A14F'}
means   = [df1_scored[df1_scored['Status'] == s]['Gut_Health_Score'].mean() for s in order]
stds    = [df1_scored[df1_scored['Status'] == s]['Gut_Health_Score'].std()  for s in order]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax2   = axes[1]
bars2 = ax2.bar(order, means, yerr=stds, capsize=6,
                color=[palette[s] for s in order], edgecolor='white', width=0.5,
                error_kw={'elinewidth': 2, 'ecolor': 'dimgrey'})

# Modified: place text above error bar tops
for bar, v, err in zip(bars2, means, stds):
    ax2.text(bar.get_x() + bar.get_width() / 2,
             v + err + 0.03,           # bar top + SD + small padding
             f'{v:.3f}',
             ha='center',
             va='bottom',                # anchor text at bottom (sits on this point)
             fontsize=11,
             fontweight='bold')

ax2.set_xticklabels(xlabels, fontsize=11, fontweight='bold')
ax2.set_ylabel('Mean Gut Health Score ± SD', fontsize=11, fontweight='bold')
ax2.set_ylim(0, 1.3)
#plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


order   = ['T1', 'T2', 'T3']
xlabels = ['T1\n(Baseline)', 'T2\n(3 months)', 'T3\n(6 months)']
#palette = {'T1': '#4E79A7', 'T2': '#F28E2B', 'T3': '#59A14F'}
means   = [df1_scored[df1_scored['Status'] == s]['Gut_Health_Score'].mean() for s in order]
stds    = [df1_scored[df1_scored['Status'] == s]['Gut_Health_Score'].std()  for s in order]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))


import matplotlib.pyplot as plt
import numpy as np

# ─── Seaborn Set1 palette (bold, saturated, publication-standard) ──────────
palette = {'T1': '#8B0000', 'T2': '#CD853F', 'T3': '#006400'}

# ─── Left: Boxplot + swarm + mean line ─────────────────────────────────────
ax = axes[0]

# 1. Boxplot with BOLD outlines and strong fill
bp = ax.boxplot(
    [df1_scored[df1_scored['Status'] == s]['Gut_Health_Score'].values
     for s in order],
    positions=[0, 1, 2],
    widths=0.35,
    patch_artist=True,
    showfliers=True,
    medianprops=dict(color='black', linewidth=2.5),           # bold median
    whiskerprops=dict(color='black', linewidth=2),              # bold whiskers
    capprops=dict(color='black', linewidth=2),                  # bold caps
    flierprops=dict(marker='o', markerfacecolor='none',
                    markeredgecolor='black', markeredgewidth=1.5,  # bold outlier outline
                    markersize=6, alpha=0.7)
)

# Color the boxes — STRONG fill, BOLD colored edges
for patch, status in zip(bp['boxes'], order):
    patch.set_facecolor(palette[status])
    patch.set_alpha(0.5)              # stronger than 0.3 — visible but points show through
    patch.set_edgecolor(palette[status])
    patch.set_linewidth(2.5)          # BOLD box outline

# 2. Individual data points — white edge for contrast against bold boxes
for i, status in enumerate(order):
    sub = df1_scored[df1_scored['Status'] == status]['Gut_Health_Score'].values
    y_sorted = np.sort(sub)
    n = len(sub)
    x_offsets = np.linspace(-0.12, 0.12, n) if n > 1 else [0]
    ax.scatter([i + xo for xo in x_offsets], y_sorted,
               color=palette[status],
               edgecolor='white',      # white edge pops against bold color
               s=40,                   # slightly larger
               zorder=5,
               alpha=0.9,
               linewidth=0.8)          # thicker point outline

# 3. Group mean with BOLD error bars
ax.errorbar(range(3), means, yerr=stds,
            fmt='o-',
            color='black',
            ecolor='black',
            elinewidth=2.5,           # BOLD error bar lines
            capsize=5,
            capthick=2.5,             # BOLD caps
            linewidth=3,              # BOLD mean line
            markersize=9,             # larger mean marker
            markeredgecolor='black',
            markeredgewidth=2,        # BOLD marker outline
            zorder=10,
            label='Group mean ± SD')

# 4. Labels and formatting
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(xlabels, fontsize=11, fontweight='bold')
ax.set_ylabel('Gut Health Score', fontsize=11, fontweight='bold')
ax.set_ylim(-0.05, 1.15)
ax.set_xlim(-0.5, 2.5)

# BOLD spines
for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(2)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.legend(fontsize=10, frameon=False, loc='upper left')
ax.tick_params(axis='both', which='major', labelsize=10, width=2)  # BOLD ticks

#ax.yaxis.grid(True, linestyle='--', alpha=0.3)
#ax.set_axisbelow(True)

In [ ]:
# Gut Health Score Trend ───────────────
from scipy import stats
import matplotlib.patches as mpatches

def pval_label(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

order   = ['T1', 'T2', 'T3']
xlabels = ['T1\n(Baseline)', 'T2\n(3 months)', 'T3\n(6 months)']
palette = {'T1': '#8B0000', 'T2': '#CD853F', 'T3': '#006400'}
means   = [df1_scored[df1_scored['Status'] == s]['Gut_Health_Score'].mean() for s in order]
stds    = [df1_scored[df1_scored['Status'] == s]['Gut_Health_Score'].std()  for s in order]
vals    = [df1_scored[df1_scored['Status'] == s]['Gut_Health_Score'].values for s in order]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), dpi=300)
fig.patch.set_facecolor('white')

# ══════════════════════════════════════════════════════
# LEFT PANEL — Box plot + jitter + significance bars
# ══════════════════════════════════════════════════════
ax = axes[0]
ax.set_facecolor('white')

bp = ax.boxplot(vals, positions=[0,1,2], widths=0.40, patch_artist=True,
                notch=False, showfliers=False,
                medianprops=dict(color='black', linewidth=2.2),
                whiskerprops=dict(color='#333333', linewidth=1.3),
                capprops=dict(color='#333333', linewidth=1.3),
                boxprops=dict(linewidth=1.3), zorder=3)
for patch, key in zip(bp['boxes'], order):
    patch.set_facecolor(palette[key] + '50'); patch.set_edgecolor(palette[key])

np.random.seed(42)
for i, (key, sub) in enumerate(zip(order, vals)):
    xj = np.random.normal(i, 0.07, len(sub))
    ax.scatter(xj, sub, color=palette[key], edgecolor='white',
               s=32, linewidth=0.4, alpha=0.85, zorder=5)

ax.plot([0,1,2], means, 'k--', lw=1.2, zorder=4, alpha=0.6)
ax.scatter([0,1,2], means, marker='D', color='white', edgecolor='black',
           s=60, linewidth=1.6, zorder=7, label='Mean')

pairs_list = [(0,1,vals[0],vals[1]),(1,2,vals[1],vals[2]),(0,2,vals[0],vals[2])]
bar_ys     = [1.07, 1.07, 1.16]
for (x1, x2, d1, d2), by in zip(pairs_list, bar_ys):
    _, p = stats.mannwhitneyu(d1, d2, alternative='two-sided')
    ax.plot([x1,x1,x2,x2], [by-0.005, by, by, by-0.005], lw=1.2, color='black')
    ax.text((x1+x2)/2, by+0.004, pval_label(p),
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks([0,1,2]); ax.set_xticklabels(xlabels, fontsize=11, fontweight='bold')
ax.set_ylabel('Gut Health Score', fontsize=12, fontweight='bold', labelpad=8)
ax.set_ylim(-0.04, 1.28); ax.set_xlim(-0.55, 2.55)
ax.tick_params(axis='y', labelsize=10, direction='out', length=3)
ax.tick_params(axis='x', length=0)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(0.9); ax.spines['bottom'].set_linewidth(0.9)
ax.yaxis.grid(False); ax.xaxis.grid(False)

handles = [mpatches.Patch(facecolor=palette[k]+'50', edgecolor=palette[k],
                           linewidth=1.3, label=k) for k in order]
handles.append(plt.Line2D([0],[0], marker='D', color='w', markerfacecolor='white',
               markeredgecolor='black', markersize=6, linewidth=0, label='Mean'))
ax.legend(handles=handles, fontsize=9, frameon=True, framealpha=0.9,
          edgecolor='#CCCCCC', loc='upper left')
#ax.text(0.01, 0.01, 'Mann–Whitney U; * p<0.05, ** p<0.01, *** p<0.001',
        #transform=ax.transAxes, fontsize=7.5, color='#555555', va='bottom', style='italic')

# ══════════════════════════════════════════════════════
# RIGHT PANEL — Box plot + mean ± SD overlay
# ══════════════════════════════════════════════════════
ax2 = axes[1]
ax2.set_facecolor('white')

bp2 = ax2.boxplot(vals, positions=[0,1,2], widths=0.40, patch_artist=True,
                  notch=False, showfliers=False,
                  medianprops=dict(color='black', linewidth=2.2),
                  whiskerprops=dict(color='#333333', linewidth=1.3),
                  capprops=dict(color='#333333', linewidth=1.3),
                  boxprops=dict(linewidth=1.3), zorder=3)
for patch, key in zip(bp2['boxes'], order):
    patch.set_facecolor(palette[key] + '50'); patch.set_edgecolor(palette[key])

for i, (m, s, key) in enumerate(zip(means, stds, order)):
    ax2.errorbar(i, m, yerr=s, fmt='none', ecolor='#333333',
                 capsize=5, capthick=1.5, elinewidth=1.5, zorder=6)
    ax2.scatter(i, m, marker='D', color='white', edgecolor='black',
                s=60, linewidth=1.6, zorder=7)
    ax2.text(i, m + s + 0.05, f'μ={m:.3f}',
             ha='center', fontsize=10, fontweight='bold', color='#222222')

ax2.set_xticks([0,1,2]); ax2.set_xticklabels(xlabels, fontsize=11, fontweight='bold')
ax2.set_ylabel('Gut Health Score', fontsize=12, fontweight='bold', labelpad=8)
ax2.set_ylim(-0.04, 1.28); ax2.set_xlim(-0.55, 2.55)
ax2.tick_params(axis='y', labelsize=10, direction='out', length=3)
ax2.tick_params(axis='x', length=0)
ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
ax2.spines['left'].set_linewidth(0.9); ax2.spines['bottom'].set_linewidth(0.9)
ax2.yaxis.grid(False); ax2.xaxis.grid(False)
ax2.text(0.01, 0.01, 'Error bars = Mean ± SD; diamond = mean',
         transform=ax2.transAxes, fontsize=7.5, color='#555555', va='bottom', style='italic')

#fig.suptitle(
    #f'Gut Health Probability Score Trend – {best_name} (Calibrated)\n'
    #'Score = P(T3)×1.0 + P(T2)×0.5 + P(T1)×0.0',
    #fontsize=12, fontweight='bold', y=1.02)

plt.tight_layout(pad=2.0)
plt.savefig('Fig4_trend_scores.png', bbox_inches='tight', dpi=300, facecolor='white')
plt.savefig('Fig4_trend_scores.pdf', bbox_inches='tight', dpi=300, facecolor='white')
plt.show()

In [ ]:
cls_palette = {'T1': '#D32F2F', 'T2': '#1976D2', 'T3': '#558B2F'}

fig, ax = plt.subplots(figsize=(7, 6), dpi=300)
ax.set_facecolor('white')

for i, cls in enumerate(classes):
    color = cls_palette[cls]
    y_cls = (y == i).astype(int)
    frac_unc, pred_unc = calibration_curve(y_cls, proba_best[:, i], n_bins=8, strategy='quantile')
    frac_cal, pred_cal = calibration_curve(y_cls, proba_cal[:, i],  n_bins=8, strategy='quantile')
    bs_unc = brier_score_loss(y_cls, proba_best[:, i])
    bs_cal = brier_score_loss(y_cls, proba_cal[:, i])

    ax.plot(pred_unc, frac_unc, 's--', color=color, lw=1.8, ms=6, alpha=0.5,
            label=f'{cls} Uncalibrated (BS={bs_unc:.3f})')
    ax.plot(pred_cal, frac_cal, 'o-',  color=color, lw=2.2, ms=7,
            label=f'{cls} Calibrated (BS={bs_cal:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1.2, alpha=0.5, label='Perfect calibration')
ax.set_xlabel('Mean predicted probability', fontsize=11, fontweight='bold')
ax.set_ylabel('Fraction of positives', fontsize=11, fontweight='bold')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.1)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=8.5, frameon=True, framealpha=0.9, edgecolor='#CCCCCC', loc='upper left')
ax.yaxis.grid(False); ax.xaxis.grid(False)

plt.tight_layout()
plt.savefig('Fig5_calibration.png', bbox_inches='tight', dpi=300, facecolor='white')
plt.show()

In [ ]:
# Apply Score to Dataset 2 which is our in-house dataset

# Dataset 2 must contain the same genera columns as Dataset 1
X2_raw = df2[feat_cols].values           # same feat_cols (non-zero genera)
X2_clr = clr_transform(X2_raw)           # same CLR transform
X2     = sel.transform(X2_clr)           # same variance threshold selector

proba2 = cal_model.predict_proba(X2)     # calibrated probabilities
scores2 = proba2 @ weights               # gut health score

df2_scored = df2[['Genus', 'Status']].copy()
df2_scored['P(T1)']            = proba2[:, 0].round(3)
df2_scored['P(T2)']            = proba2[:, 1].round(3)
df2_scored['P(T3)']            = proba2[:, 2].round(3)
df2_scored['Gut_Health_Score'] = scores2.round(3)
df2_scored['Predicted_Status'] = le.inverse_transform(proba2.argmax(axis=1))
df2_scored.to_csv('Table3_Dataset2_probability_scores.csv', index=False)

print('Dataset 2 – Gut Health Scores:')
display(df2_scored)
print(f'\nDS2 mean score: {scores2.mean():.3f}')
print(f'DS1-T2 mean  : {df1_scored[df1_scored["Status"]=="T2"]["Gut_Health_Score"].mean():.3f}')

In [ ]:
# Dataset 2 must contain the same genera columns as Dataset 1
X2_raw = df2[feat_cols].values           # same feat_cols (non-zero genera)
X2_clr = clr_transform(X2_raw)           # same CLR transform
X2     = sel.transform(X2_clr)           # same variance threshold selector

proba2 = cal_model.predict_proba(X2)     # calibrated probabilities
scores2 = proba2 @ weights               # gut health score

df2_scored = df2[['Genus', 'Status']].copy()
df2_scored['P(T1)']            = proba2[:, 0].round(3)
df2_scored['P(T2)']            = proba2[:, 1].round(3)
df2_scored['P(T3)']            = proba2[:, 2].round(3)
df2_scored['Gut_Health_Score'] = scores2.round(3)
df2_scored['Predicted_Status'] = le.inverse_transform(proba2.argmax(axis=1))
df2_scored.to_csv('Table3_Dataset2_probability_scores.csv', index=False)

print('Dataset 2 – Gut Health Scores:')
display(df2_scored)
print(f'\nDS2 mean score: {scores2.mean():.3f}')
print(f'DS1-T2 mean  : {df1_scored[df1_scored["Status"]=="T2"]["Gut_Health_Score"].mean():.3f}')

In [ ]:
# Dataset 2 vs Dataset 1 Treatment Trajectory

d1_t1 = df1_scored[df1_scored['Status'] == 'T1']['Gut_Health_Score'].values
d1_t2 = df1_scored[df1_scored['Status'] == 'T2']['Gut_Health_Score'].values
d1_t3 = df1_scored[df1_scored['Status'] == 'T3']['Gut_Health_Score'].values

groups  = [d1_t1, d1_t2, scores2, d1_t3]
glabels = [
    f'DS1-T1\n(n={len(d1_t1)})',
    f'DS1-T2\n(n={len(d1_t2)})',
    f'DS2-T2\n(n={len(scores2)})',
    f'DS1-T3\n(n={len(d1_t3)})'
]
gcols = ['#EF5350', '#FFA726', '#AB47BC', '#66BB6A']

fig, ax = plt.subplots(figsize=(10, 5.5))
bp = ax.boxplot(groups, labels=glabels, patch_artist=True, widths=0.5,
                medianprops=dict(color='black', lw=2.5), showfliers=False)
for patch, col in zip(bp['boxes'], gcols):
    patch.set_facecolor(col); patch.set_alpha(0.6)
for i, (data, col) in enumerate(zip(groups, gcols), 1):
    ax.scatter(np.random.normal(i, 0.08, len(data)), data,
               color=col, edgecolor='white', s=40, zorder=5, alpha=0.9)
for i, data in enumerate(groups, 1):
    ax.text(i, max(data) + 0.04, f'μ={np.mean(data):.3f}',
            ha='center', fontsize=9, fontweight='bold')
ax.set_ylabel('Gut Health Score', fontsize=11)
ax.set_title('Dataset 2 Patients vs Dataset 1 Treatment Trajectory\n'
             '(Calibrated SVM Gut Health Score)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('Fig6_DS2_comparison.png', bbox_inches='tight', dpi=200)
plt.show()

In [ ]:
# T1 vs T3 Score Shift Summary

t1_df = df1_scored[df1_scored['Status'] == 'T1']
t3_df = df1_scored[df1_scored['Status'] == 'T3']
delta = t3_df['Gut_Health_Score'].mean() - t1_df['Gut_Health_Score'].mean()

shift_df = pd.DataFrame({
    'Time Point': ['T1 (Day 0)', 'T3 (6 months)'],
    'n': [len(t1_df), len(t3_df)],
    'Mean Score': [round(t1_df['Gut_Health_Score'].mean(), 3),
                   round(t3_df['Gut_Health_Score'].mean(), 3)],
    'SD':         [round(t1_df['Gut_Health_Score'].std(), 3),
                   round(t3_df['Gut_Health_Score'].std(), 3)],
    'Median':     [round(t1_df['Gut_Health_Score'].median(), 3),
                   round(t3_df['Gut_Health_Score'].median(), 3)],
    'Change vs T1': ['—', f'{delta:+.3f}']
})
shift_df.to_csv('Table4_T1vsT3_shift.csv', index=False)
display(shift_df)
direction = 'IMPROVEMENT' if delta > 0 else 'DEGRADATION'
print(f'\n→ Gut health score {direction} of {abs(delta):.3f} points from T1 to T3')

**We actually realized that Gut health score IMPROVEMENT was 0.253 points from T1 to T3**